# Фінальний проєкт. Датасет infectious_cases

PostgreSQL 16 у Colab через pgserver. Файл `infectious_cases.csv` читається напряму з мого репозиторію на GitHub, тому нічого завантажувати вручну не треба.

## 0. Підготовка середовища

In [1]:
%pip install -q pixeltable-pgserver==0.5.0 psycopg2-binary sqlalchemy pandas jupysql

In [2]:
# імпорти
import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

%load_ext sql

In [3]:
pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://')
engine = create_engine(uri)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

%sql engine
%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False

PostgreSQL 16.10 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-10), 64-bit


## 1. Завантаження даних

Створюю схему `pandemic` і роблю її схемою за замовчуванням.

In [4]:
%%sql
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

SHOW search_path;

1 rows affected.

search_path
pandemic


Імпорт CSV так само, як у домашніх: читаю файл через pandas (з GitHub, raw-посилання) і записую в PostgreSQL через `to_sql`.
Усі колонки читаю як текст (`dtype=str`), а порожні клітинки залишаю порожніми рядками, тобто таблиця `infectious_cases` - це сирі дані як у файлі.
Назви колонок переводжу в нижній регістр, щоб у SQL не писати їх у лапках.

In [5]:
url = 'https://raw.githubusercontent.com/sergejbrutkovskij9-tech/goit-rdb-fp/main/infectious_cases.csv'
df = pd.read_csv(url, dtype=str, keep_default_na=False)
df.columns = df.columns.str.lower()
print(df.shape)

df.to_sql('infectious_cases', engine, schema='pandemic', if_exists='replace', index=False)
print('таблицю infectious_cases завантажено')

(10521, 12)


таблицю infectious_cases завантажено


In [6]:
%%sql
SELECT COUNT(*) AS n_rows
FROM infectious_cases;

1 rows affected.

n_rows
10521


In [7]:
%%sql
SELECT *
FROM infectious_cases
LIMIT 10;

10 rows affected.

entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
Afghanistan,AFG,1980,,6160,0,,,,,,
Afghanistan,AFG,1981,,5859,0,,,,,,
Afghanistan,AFG,1982,,9730,0,,,,,,
Afghanistan,AFG,1983,,13937,0,,,,,,
Afghanistan,AFG,1984,,3864,0,,,,,,
Afghanistan,AFG,1985,,13867,0,,,,,,
Afghanistan,AFG,1986,,12901,0,,,,,,
Afghanistan,AFG,1987,,4396,0,,,,,,
Afghanistan,AFG,1988,,2149,0,,,,,,
Afghanistan,AFG,1989,,385,0,,,,,,


In [8]:
%%sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name = 'infectious_cases'
ORDER BY ordinal_position;

12 rows affected.

column_name,data_type
entity,text
code,text
year,text
number_yaws,text
polio_cases,text
cases_guinea_worm,text
number_rabies,text
number_malaria,text
number_hiv,text
number_tuberculosis,text


Скільки порожніх значень у колонках з кількістю хворих і скільки рядків без коду країни:

In [9]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE code = '')          AS empty_code,
    COUNT(*) FILTER (WHERE number_rabies = '') AS empty_rabies,
    COUNT(*) FILTER (WHERE number_malaria = '') AS empty_malaria,
    COUNT(*) FILTER (WHERE polio_cases = '')   AS empty_polio,
    COUNT(DISTINCT entity)                     AS n_entities,
    MIN(year)                                  AS min_year,
    MAX(year)                                  AS max_year
FROM infectious_cases;

1 rows affected.

empty_code,empty_rabies,empty_malaria,empty_polio,n_entities,min_year,max_year
1128,3741,3741,2911,245,1920,2023


У 1128 рядках порожній `code` - це регіони і групи країн (Africa, World і т.д.), а не окремі країни.

## 2. Нормалізація до 3НФ

У сирій таблиці `entity` і `code` повторюються в кожному рядку (для кожного року). `code` залежить від `entity`, а не від пари (`entity`, `year`), тому це надлишкові дані.
Розбиваю на дві таблиці:

- `entities` - унікальні комбінації `entity` + `code` зі своїм `id`;
- `infectious_cases_norm` - показники захворювань по роках, де замість назви і коду стоїть `entity_id`.

Заодно порожні рядки перетворюю на `NULL`, а значення - на числа.

In [10]:
%%sql
DROP TABLE IF EXISTS infectious_cases_norm;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    id     SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code   TEXT,
    UNIQUE (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(code, '')
FROM infectious_cases
ORDER BY entity;

CREATE TABLE infectious_cases_norm (
    id                   SERIAL PRIMARY KEY,
    entity_id            INTEGER NOT NULL REFERENCES entities(id),
    year                 INTEGER NOT NULL,
    number_yaws          NUMERIC,
    polio_cases          NUMERIC,
    cases_guinea_worm    NUMERIC,
    number_rabies        NUMERIC,
    number_malaria       NUMERIC,
    number_hiv           NUMERIC,
    number_tuberculosis  NUMERIC,
    number_smallpox      NUMERIC,
    number_cholera_cases NUMERIC,
    UNIQUE (entity_id, year)
);

INSERT INTO infectious_cases_norm (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases
)
SELECT
    e.id,
    ic.year::INTEGER,
    NULLIF(ic.number_yaws, '')::NUMERIC,
    NULLIF(ic.polio_cases, '')::NUMERIC,
    NULLIF(ic.cases_guinea_worm, '')::NUMERIC,
    NULLIF(ic.number_rabies, '')::NUMERIC,
    NULLIF(ic.number_malaria, '')::NUMERIC,
    NULLIF(ic.number_hiv, '')::NUMERIC,
    NULLIF(ic.number_tuberculosis, '')::NUMERIC,
    NULLIF(ic.number_smallpox, '')::NUMERIC,
    NULLIF(ic.number_cholera_cases, '')::NUMERIC
FROM infectious_cases ic
JOIN entities e
    ON e.entity = ic.entity
   AND e.code IS NOT DISTINCT FROM NULLIF(ic.code, '')
ORDER BY e.id, ic.year::INTEGER;

245 rows affected.

10521 rows affected.

++
||
++
++

Перевірка: кількість рядків має збігатися з сирою таблицею.

In [11]:
%%sql
SELECT
    (SELECT COUNT(*) FROM infectious_cases)      AS raw_rows,
    (SELECT COUNT(*) FROM infectious_cases_norm) AS norm_rows,
    (SELECT COUNT(*) FROM entities)              AS n_entities;

1 rows affected.

raw_rows,norm_rows,n_entities
10521,10521,245


In [12]:
%%sql
SELECT *
FROM entities
ORDER BY id
LIMIT 10;

10 rows affected.

id,entity,code
1,Afghanistan,AFG
2,Africa,None
3,African Region (WHO),None
4,Albania,ALB
5,Algeria,DZA
6,American Samoa,ASM
7,Americas,None
8,Andorra,AND
9,Angola,AGO
10,Antigua and Barbuda,ATG


In [13]:
%%sql
SELECT *
FROM infectious_cases_norm
ORDER BY id
LIMIT 10;

10 rows affected.

id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
1,1,1960,None,None,None,None,None,None,None,None,887
2,1,1965,None,None,None,None,None,None,None,None,218
3,1,1980,None,6160,0,None,None,None,None,None,None
4,1,1981,None,5859,0,None,None,None,None,None,None
5,1,1982,None,9730,0,None,None,None,None,None,None
6,1,1983,None,13937,0,None,None,None,None,None,None
7,1,1984,None,3864,0,None,None,None,None,None,None
8,1,1985,None,13867,0,None,None,None,None,None,None
9,1,1986,None,12901,0,None,None,None,None,None,None
10,1,1987,None,4396,0,None,None,None,None,None,None


## 3. Аналіз Number_rabies

Для кожної комбінації `entity` + `code` рахую середнє, мінімум, максимум і суму.
Рядки, де значення немає (NULL), не враховуються. Сортування за середнім за спаданням, 10 рядків.

In [14]:
%%sql
SELECT
    e.id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies
FROM infectious_cases_norm n
JOIN entities e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;

10 rows affected.

id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11
121,Lower Middle Income (WB),None,15193.96,10202.53,19182.80,455818.78
202,South Asia (WB),None,11729.89,7271.28,15361.88,351896.66
206,South-East Asia Region (WHO),None,11424.33,6806.01,15641.96,342729.81
77,G20,None,10189.05,6339.08,13164.88,305671.38
97,India,IND,8599.17,5425.87,11121.14,257975.20
209,Sub-Saharan Africa (WB),None,4748.27,4356.82,5017.98,142448.04
3,African Region (WHO),None,4649.70,4283.07,4917.78,139491.07
120,Low Income (WB),None,3568.11,3014.62,3873.53,107043.16
58,East Asia & Pacific (WB),None,3427.46,2009.94,4591.01,102823.83


У топі є не тільки країни, а й регіони (World, групи країн за доходом), бо вони теж є в датасеті як окремі `entity`. У них `code` порожній, крім World (`OWID_WRL`).

## 4. Колонка різниці в роках

- `start_date` - перше січня року: `MAKE_DATE(year, 1, 1)`;
- `cur_date` - поточна дата: `CURRENT_DATE`;
- `year_diff` - різниця в роках: `AGE()` дає інтервал між датами, а `DATE_PART('year', ...)` бере з нього кількість повних років.

In [15]:
%%sql
SELECT
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1) AS start_date,
    CURRENT_DATE AS cur_date,
    DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS year_diff
FROM infectious_cases_norm n
JOIN entities e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;

10 rows affected.

entity,year,start_date,cur_date,year_diff
Afghanistan,1960,1960-01-01,2026-11-16,66
Afghanistan,1965,1965-01-01,2026-11-16,61
Afghanistan,1980,1980-01-01,2026-11-16,46
Afghanistan,1981,1981-01-01,2026-11-16,45
Afghanistan,1982,1982-01-01,2026-11-16,44
Afghanistan,1983,1983-01-01,2026-11-16,43
Afghanistan,1984,1984-01-01,2026-11-16,42
Afghanistan,1985,1985-01-01,2026-11-16,41
Afghanistan,1986,1986-01-01,2026-11-16,40
Afghanistan,1987,1987-01-01,2026-11-16,39


## 5. Власні функції

### Функція 1. Різниця в роках

Приймає рік, будує з нього 1 січня і повертає різницю в роках з поточною датою. Логіка та сама, що в завданні 4.

In [16]:
%%sql
DROP FUNCTION IF EXISTS year_diff(INTEGER);

CREATE FUNCTION year_diff(y INTEGER)
RETURNS INTEGER
LANGUAGE sql
AS $$
    SELECT DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(y, 1, 1)))::INTEGER;
$$;

SELECT year_diff(1996) AS diff_1996, year_diff(2000) AS diff_2000;

1 rows affected.

diff_1996,diff_2000
30,26


Використання в запиті. Для перевірки поруч колонка з завдання 4, значення мають збігатися.

In [17]:
%%sql
SELECT
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1) AS start_date,
    year_diff(n.year) AS year_diff_func,
    DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS year_diff_builtin
FROM infectious_cases_norm n
JOIN entities e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;

10 rows affected.

entity,year,start_date,year_diff_func,year_diff_builtin
Afghanistan,1960,1960-01-01,66,66
Afghanistan,1965,1965-01-01,61,61
Afghanistan,1980,1980-01-01,46,46
Afghanistan,1981,1981-01-01,45,45
Afghanistan,1982,1982-01-01,44,44
Afghanistan,1983,1983-01-01,43,43
Afghanistan,1984,1984-01-01,42,42
Afghanistan,1985,1985-01-01,41,41
Afghanistan,1986,1986-01-01,40,40
Afghanistan,1987,1987-01-01,39,39


In [18]:
%%sql
SELECT COUNT(*) AS n_mismatch
FROM infectious_cases_norm
WHERE year_diff(year) <> DATE_PART('year', AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER;

1 rows affected.

n_mismatch
0


### Функція 2. Середня кількість захворювань за період

Приймає кількість захворювань за рік і дільник, повертає середнє за період. Якщо дільник 0, функція повертає NULL, щоб не було помилки ділення на нуль.

In [19]:
%%sql
DROP FUNCTION IF EXISTS avg_cases(NUMERIC, NUMERIC);

CREATE FUNCTION avg_cases(cases NUMERIC, divider NUMERIC)
RETURNS NUMERIC
LANGUAGE sql
AS $$
    SELECT ROUND(cases / NULLIF(divider, 0), 2);
$$;

SELECT avg_cases(120, 12) AS per_month, avg_cases(120, 4) AS per_quarter, avg_cases(120, 2) AS per_half, avg_cases(120, 0) AS div_zero;

1 rows affected.

per_month,per_quarter,per_half,div_zero
10.00,30.00,60.00,None


Використання на `number_rabies`. Беру тільки записи, де значення числове: у сирій таблиці це непорожні рядки,
а в нормалізованій - `IS NOT NULL` (порожні рядки при нормалізації стали NULL).

In [20]:
%%sql
SELECT
    e.entity,
    n.year,
    n.number_rabies,
    avg_cases(n.number_rabies, 12) AS rabies_per_month,
    avg_cases(n.number_rabies, 4)  AS rabies_per_quarter,
    avg_cases(n.number_rabies, 2)  AS rabies_per_half_year
FROM infectious_cases_norm n
JOIN entities e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
ORDER BY n.id
LIMIT 10;

10 rows affected.

entity,year,number_rabies,rabies_per_month,rabies_per_quarter,rabies_per_half_year
Afghanistan,1990,2.145772,0.18,0.54,1.07
Afghanistan,1991,2.3209748,0.19,0.58,1.16
Afghanistan,1992,2.6180408,0.22,0.65,1.31
Afghanistan,1993,2.8531172,0.24,0.71,1.43
Afghanistan,1994,2.9899871,0.25,0.75,1.49
Afghanistan,1995,3.126361,0.26,0.78,1.56
Afghanistan,1996,3.281437,0.27,0.82,1.64
Afghanistan,1997,3.446122,0.29,0.86,1.72
Afghanistan,1998,3.5570247,0.30,0.89,1.78
Afghanistan,1999,3.607037,0.30,0.90,1.80


Те саме на сирій таблиці: фільтр по регулярному виразу залишає тільки рядки, де в `number_rabies` число.

In [21]:
%%sql
SELECT
    entity,
    year,
    number_rabies,
    avg_cases(number_rabies::NUMERIC, 12) AS rabies_per_month,
    avg_cases(number_rabies::NUMERIC, 4)  AS rabies_per_quarter,
    avg_cases(number_rabies::NUMERIC, 2)  AS rabies_per_half_year
FROM infectious_cases
WHERE number_rabies ~ '^[0-9]+(\.[0-9]+)?$'
ORDER BY entity, year::INTEGER
LIMIT 10;

10 rows affected.

entity,year,number_rabies,rabies_per_month,rabies_per_quarter,rabies_per_half_year
Afghanistan,1990,2.145772,0.18,0.54,1.07
Afghanistan,1991,2.3209748,0.19,0.58,1.16
Afghanistan,1992,2.6180408,0.22,0.65,1.31
Afghanistan,1993,2.8531172,0.24,0.71,1.43
Afghanistan,1994,2.9899871,0.25,0.75,1.49
Afghanistan,1995,3.126361,0.26,0.78,1.56
Afghanistan,1996,3.281437,0.27,0.82,1.64
Afghanistan,1997,3.446122,0.29,0.86,1.72
Afghanistan,1998,3.5570247,0.30,0.89,1.78
Afghanistan,1999,3.607037,0.30,0.90,1.80


## Висновки

1. Дані імпортовані в схему `pandemic` у сирому вигляді: 10521 рядок, 245 комбінацій `entity` + `code`, роки з 1920 по 2023.
2. Таблиця нормалізована до 3НФ: назви і коди країн зберігаються один раз у `entities`, а показники по роках - в `infectious_cases_norm` з посиланням `entity_id`. Порожні значення стали `NULL`, а текст - числами, тому з ними можна рахувати агрегати.
3. Найбільше середнє `number_rabies` у World і великих групах країн, серед окремих країн на першому місці Індія.
4. Різницю в роках можна порахувати вбудованими функціями `MAKE_DATE`, `CURRENT_DATE`, `AGE` і `DATE_PART`. Власна функція `year_diff` дає той самий результат для всіх рядків.
5. Функція `avg_cases` ділить річну кількість на 12, 4 або 2 і дає середнє за місяць, квартал і півріччя.